In [1]:
import json
from typing import Dict, List, Any

def parse_study_overview(protocol: Dict) -> Dict[str, Any]:
    """Study Overview 섹션 파싱"""
    identification = protocol.get('identificationModule', {})
    description = protocol.get('descriptionModule', {})
    status = protocol.get('statusModule', {})
    design = protocol.get('designModule', {})
    conditions = protocol.get('conditionsModule', {})
    interventions_module = protocol.get('armsInterventionsModule', {})
    sponsor = protocol.get('sponsorCollaboratorsModule', {})
    
    overview = {
        'NCT ID': identification.get('nctId', 'N/A'),
        'Organization Study ID': identification.get('orgStudyIdInfo', {}).get('id', 'N/A'),
        'Brief Title': identification.get('briefTitle', 'N/A'),
        'Official Title': identification.get('officialTitle', 'N/A'),
        'Brief Summary': description.get('briefSummary', 'N/A'),
        'Detailed Description': description.get('detailedDescription', 'N/A'),
        'Conditions': conditions.get('conditions', []),
        'Study Type': design.get('studyType', 'N/A'),
        'Phases': design.get('phases', []),
        'Enrollment': design.get('enrollmentInfo', {}).get('count', 'N/A'),
        'Enrollment Type': design.get('enrollmentInfo', {}).get('type', 'N/A'),
        'Overall Status': status.get('overallStatus', 'N/A'),
        'Start Date': status.get('startDateStruct', {}).get('date', 'N/A'),
        'Primary Completion Date': status.get('primaryCompletionDateStruct', {}).get('date', 'N/A'),
        'Completion Date': status.get('completionDateStruct', {}).get('date', 'N/A'),
        'Lead Sponsor': sponsor.get('leadSponsor', {}).get('name', 'N/A'),
        'Responsible Party': sponsor.get('responsibleParty', {}),
        'Interventions': []
    }
    
    # Interventions 파싱
    for intervention in interventions_module.get('interventions', []):
        overview['Interventions'].append({
            'Type': intervention.get('type', 'N/A'),
            'Name': intervention.get('name', 'N/A'),
            'Description': intervention.get('description', 'N/A'),
            'Arm Group Labels': intervention.get('armGroupLabels', [])
        })
    
    return overview


def parse_participation_criteria(protocol: Dict) -> Dict[str, Any]:
    """Participation Criteria 섹션 파싱"""
    eligibility = protocol.get('eligibilityModule', {})
    
    criteria = {
        'Eligibility Criteria': eligibility.get('eligibilityCriteria', 'N/A'),
        'Healthy Volunteers': 'Yes' if eligibility.get('healthyVolunteers', False) else 'No',
        'Sex': eligibility.get('sex', 'N/A'),
        'Minimum Age': eligibility.get('minimumAge', 'N/A'),
        'Maximum Age': eligibility.get('maximumAge', 'N/A'),
        'Standard Ages': eligibility.get('stdAges', [])
    }
    
    return criteria


def parse_study_plan(protocol: Dict) -> Dict[str, Any]:
    """Study Plan 섹션 파싱"""
    design = protocol.get('designModule', {})
    design_info = design.get('designInfo', {})
    arms_module = protocol.get('armsInterventionsModule', {})
    outcomes = protocol.get('outcomesModule', {})
    
    plan = {
        'Design Information': {
            'Allocation': design_info.get('allocation', 'N/A'),
            'Intervention Model': design_info.get('interventionModel', 'N/A'),
            'Intervention Model Description': design_info.get('interventionModelDescription', 'N/A'),
            'Primary Purpose': design_info.get('primaryPurpose', 'N/A'),
            'Masking': design_info.get('maskingInfo', {}).get('masking', 'N/A'),
            'Masking Description': design_info.get('maskingInfo', {}).get('maskingDescription', 'N/A'),
            'Who Masked': design_info.get('maskingInfo', {}).get('whoMasked', [])
        },
        'Arm Groups': [],
        'Primary Outcomes': [],
        'Secondary Outcomes': [],
        'Other Outcomes': []
    }
    
    # Arm Groups 파싱
    for arm in arms_module.get('armGroups', []):
        plan['Arm Groups'].append({
            'Label': arm.get('label', 'N/A'),
            'Type': arm.get('type', 'N/A'),
            'Description': arm.get('description', 'N/A'),
            'Intervention Names': arm.get('interventionNames', [])
        })
    
    # Primary Outcomes 파싱
    for outcome in outcomes.get('primaryOutcomes', []):
        plan['Primary Outcomes'].append({
            'Measure': outcome.get('measure', 'N/A'),
            'Description': outcome.get('description', 'N/A'),
            'Time Frame': outcome.get('timeFrame', 'N/A')
        })
    
    # Secondary Outcomes 파싱
    for outcome in outcomes.get('secondaryOutcomes', []):
        plan['Secondary Outcomes'].append({
            'Measure': outcome.get('measure', 'N/A'),
            'Description': outcome.get('description', 'N/A'),
            'Time Frame': outcome.get('timeFrame', 'N/A')
        })
    
    # Other Outcomes 파싱
    for outcome in outcomes.get('otherOutcomes', []):
        plan['Other Outcomes'].append({
            'Measure': outcome.get('measure', 'N/A'),
            'Description': outcome.get('description', 'N/A'),
            'Time Frame': outcome.get('timeFrame', 'N/A')
        })
    
    return plan


def print_section(title: str, data: Dict[str, Any], indent: int = 0):
    """섹션 데이터를 보기 좋게 출력"""
    indent_str = "  " * indent
    print(f"\n{indent_str}{'='*80}")
    print(f"{indent_str}{title}")
    print(f"{indent_str}{'='*80}")
    
    for key, value in data.items():
        if isinstance(value, dict):
            print(f"\n{indent_str}[{key}]")
            print_section("", value, indent + 1)
        elif isinstance(value, list):
            if len(value) == 0:
                print(f"{indent_str}{key}: (없음)")
            elif isinstance(value[0], dict):
                print(f"\n{indent_str}[{key}]")
                for i, item in enumerate(value, 1):
                    print(f"{indent_str}  #{i}")
                    for k, v in item.items():
                        if isinstance(v, list):
                            print(f"{indent_str}    {k}: {', '.join(map(str, v)) if v else '(없음)'}")
                        else:
                            print(f"{indent_str}    {k}: {v}")
            else:
                print(f"{indent_str}{key}: {', '.join(map(str, value))}")
        else:
            # 긴 텍스트는 줄바꿈 처리
            if isinstance(value, str) and len(value) > 100:
                print(f"{indent_str}{key}:")
                # 텍스트를 80자씩 끊어서 출력
                lines = value.split('\n')
                for line in lines:
                    if line.strip():
                        print(f"{indent_str}  {line.strip()}")
            else:
                print(f"{indent_str}{key}: {value}")


def parse_clinical_trial(json_file: str):
    """Clinical Trial JSON 파일 전체 파싱"""
    # JSON 파일 읽기
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    protocol = data.get('protocolSection', {})
    
    # 각 섹션 파싱
    study_overview = parse_study_overview(protocol)
    participation_criteria = parse_participation_criteria(protocol)
    study_plan = parse_study_plan(protocol)
    
    # 결과 출력
    print("\n" + "="*80)
    print("CLINICAL TRIAL DATA PARSER")
    print("="*80)
    
    print_section("📋 STUDY OVERVIEW", study_overview)
    print_section("👥 PARTICIPATION CRITERIA", participation_criteria)
    print_section("📊 STUDY PLAN", study_plan)
    
    # 파싱된 데이터를 딕셔너리로 반환
    return {
        'Study Overview': study_overview,
        'Participation Criteria': participation_criteria,
        'Study Plan': study_plan
    }


def save_parsed_data(parsed_data: Dict, output_file: str = 'pasred_data/pased_section_1.json'):
    """파싱된 데이터를 JSON 파일로 저장"""
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(parsed_data, f, ensure_ascii=False, indent=2)
    print(f"\n✅ 파싱된 데이터가 '{output_file}'에 저장되었습니다.")


# 사용 예시
if __name__ == "__main__":
    json_file = "NCT04882579.json"  # 또는 "NCT07029308.json"
    
    # 데이터 파싱 및 출력
    parsed_data = parse_clinical_trial(json_file)
    
    # 파싱된 데이터 저장
    save_parsed_data(parsed_data, "pased_section_1.json")


CLINICAL TRIAL DATA PARSER

📋 STUDY OVERVIEW
NCT ID: NCT04882579
Organization Study ID: PulmonaryEmbolismPoCUSOdense
Brief Title: Point-of-care Ultrasound in Suspected Pulmonary Embolism
Official Title:
  Point-of-care Ultrasound in the Diagnostic Work-up of Suspected Pulmonary Embolism - a Multicenter Randomized Controlled Trial
Brief Summary:
  Pulmonary embolism (PE) is a common cardiovascular condition with an estimated incidence of 0.60 to 1.12 per 1000 inhabitants in the United States of America, and the diagnosis is challenging as patients with PE present with a wide array of symptoms.
  Computed tomography pulmonary angriography (CTPA) and lung ventilation-perfusion scintigraphy (VQ) are considered the gold-standards in PE-diagnostics but may not always be feasible. CTPA is contraindicated by contrast allergy or renal failure and both modalities require involvement of multiple staff-members and transport of the patient. Lung scintigraphy cannot be performed in an emergency sit

In [1]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Qwen/Qwen3-Embedding-4B",  # 원하는 모델 이름
    local_dir="/data/home/sss2061/2025스마트임상/Qwen3-Embedding-4b"  # 원하는 저장 경로
)

snapshot_download(
    repo_id="Qwen/Qwen3-Embedding-8B",  # 원하는 모델 이름
    local_dir="/data/home/sss2061/2025스마트임상/Qwen3-Embedding-8b"  # 원하는 저장 경로
)

/data/home/bmi-lab/anaconda3/envs/jy3/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 17 files: 100%|██████████| 17/17 [26:51<00:00, 94.78s/it]


'/data/home/sss2061/2025스마트임상/Qwen3-Embedding-8b'

In [2]:
snapshot_download(
    repo_id="Xiaojian9992024/Qwen2.5-Dyanka-7B-Preview",  # 원하는 모델 이름
    local_dir="/data/home/sss2061/2025스마트임상/Qwen2.5-Dyanka-7B-Preview"  # 원하는 저장 경로
)
meta-llama/Llama-3.1-8B-Instruct

Fetching 16 files: 100%|██████████| 16/16 [26:39<00:00, 99.94s/it]


'/data/home/sss2061/2025스마트임상/Qwen2.5-Dyanka-7B-Preview'

# 